# Benchmark de Modelo de Email Phishing (DistilBERT)

**Dataset:** `cybersectony/PhishingEmailDetectionv2.0` (HuggingFace)
**Entrada:** Texto do email (subject + body)
**Hardware:** Google Colab T4 (16 GB VRAM)

**Modelos avaliados:**
- DistilBERT Email (`cybersectony/phishing-email-detection-distilbert_v2.4.1`) — ~66M params, fine-tuned para phishing email
- XLM-RoBERTa (`xlm-roberta-base`) — ~278M params, multilingual, fine-tuned no mesmo dataset para comparação

**Tradução:** MarianMT (`Helsinki-NLP/opus-mt-tc-big-pt-en`) para emails em português

**Métricas:** Accuracy, F1, Precision, Recall, AUC-ROC, Confusion Matrix, Latência

In [ ]:
# ============================================================
# 1. Setup — Verificação de GPU e Instalação
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'AVISO: GPU não detectada!')

!pip install -q transformers datasets langdetect scikit-learn matplotlib seaborn sentencepiece
print('Instalação concluída!')

In [ ]:
# ============================================================
# 2. Carregar Dataset cybersectony/PhishingEmailDetectionv2.0
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time, os, gc, warnings
warnings.filterwarnings('ignore')

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    MarianMTModel, MarianTokenizer
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve,
    precision_recall_curve, average_precision_score
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Carregar dataset
ds = load_dataset('cybersectony/PhishingEmailDetectionv2.0')
print(f'Dataset carregado: {ds}')

# Usar split de teste se disponível, senão sample do train
if 'test' in ds:
    df = ds['test'].to_pandas()
else:
    df = ds['train'].to_pandas()
    # Amostra para benchmark (se dataset for muito grande)
    if len(df) > 5000:
        df = df.sample(n=5000, random_state=42).reset_index(drop=True)

print(f'Amostras para avaliação: {len(df)}')
print(f'Colunas: {list(df.columns)}')
print(f'\nDistribuição de labels:')
print(df['label'].value_counts() if 'label' in df.columns else df.iloc[:, -1].value_counts())

In [ ]:
# ============================================================
# 3. Avaliar DistilBERT Email (Baseline)
#    cybersectony/phishing-email-detection-distilbert_v2.4.1
# ============================================================
MODEL_ID = 'cybersectony/phishing-email-detection-distilbert_v2.4.1'

tokenizer_distilbert = AutoTokenizer.from_pretrained(MODEL_ID)
model_distilbert = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device).eval()
print(f'Modelo carregado: {MODEL_ID}')
print(f'Parâmetros: {sum(p.numel() for p in model_distilbert.parameters()):,}')

# Mapear labels do dataset para binário (0=legit, 1=phishing)
text_col = 'text' if 'text' in df.columns else df.columns[0]
label_col = 'label' if 'label' in df.columns else df.columns[-1]

# Verificar mapeamento de labels
unique_labels = df[label_col].unique()
print(f'Labels únicos: {unique_labels}')

# Mapear labels string para int se necessário
if df[label_col].dtype == object:
    label_map = {}
    for lbl in unique_labels:
        lbl_lower = str(lbl).lower()
        if 'phish' in lbl_lower or 'spam' in lbl_lower or 'malicious' in lbl_lower:
            label_map[lbl] = 1
        else:
            label_map[lbl] = 0
    print(f'Mapeamento de labels: {label_map}')
    y_true = df[label_col].map(label_map).values
else:
    y_true = df[label_col].values

# Inferência em batches
BATCH_SIZE = 32
y_probs_distilbert = []
latencies_distilbert = []
texts = df[text_col].tolist()

print(f'\nIniciando inferência ({len(texts)} amostras, batch={BATCH_SIZE})...')
for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i:i+BATCH_SIZE]
    inputs = tokenizer_distilbert(
        batch_texts, return_tensors='pt', truncation=True,
        max_length=512, padding=True
    ).to(device)
    
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model_distilbert(**inputs)
    t1 = time.perf_counter()
    
    probs = torch.softmax(outputs.logits, dim=-1)
    # Phishing é label 1
    phishing_probs = probs[:, 1].cpu().numpy()
    y_probs_distilbert.extend(phishing_probs)
    latencies_distilbert.append((t1 - t0) / len(batch_texts) * 1000)  # ms per sample
    
    if (i // BATCH_SIZE) % 20 == 0:
        print(f'  Batch {i//BATCH_SIZE + 1}/{(len(texts)-1)//BATCH_SIZE + 1}')

y_probs_distilbert = np.array(y_probs_distilbert)
y_pred_distilbert = (y_probs_distilbert > 0.5).astype(int)

print(f'\n--- Resultados DistilBERT Email ---')
print(classification_report(y_true, y_pred_distilbert, target_names=['Legítimo', 'Phishing']))
print(f'AUC-ROC: {roc_auc_score(y_true, y_probs_distilbert):.4f}')
print(f'Latência média: {np.mean(latencies_distilbert):.2f} ms/amostra')
print(f'Latência P50: {np.percentile(latencies_distilbert, 50):.2f} ms/amostra')
print(f'Latência P95: {np.percentile(latencies_distilbert, 95):.2f} ms/amostra')

# Liberar memória
del model_distilbert, tokenizer_distilbert
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# 4. Avaliar Alternativa Multilingual (XLM-RoBERTa)
# ============================================================
# Fine-tune rápido de XLM-R no mesmo dataset para comparação
from transformers import TrainingArguments, Trainer
from torch.utils.data import Dataset as TorchDataset

XLMR_MODEL_ID = 'xlm-roberta-base'

tokenizer_xlmr = AutoTokenizer.from_pretrained(XLMR_MODEL_ID)
model_xlmr = AutoModelForSequenceClassification.from_pretrained(
    XLMR_MODEL_ID, num_labels=2
).to(device).eval()
print(f'Modelo carregado: {XLMR_MODEL_ID}')
print(f'Parâmetros: {sum(p.numel() for p in model_xlmr.parameters()):,}')

# Dataset para fine-tuning rápido
class EmailDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# Split train/eval para fine-tuning
from sklearn.model_selection import train_test_split
train_texts, eval_texts, train_labels, eval_labels = train_test_split(
    texts, y_true.tolist(), test_size=0.2, random_state=42, stratify=y_true
)

train_ds = EmailDataset(train_texts, train_labels, tokenizer_xlmr)
eval_ds = EmailDataset(eval_texts, eval_labels, tokenizer_xlmr)

# Fine-tuning rápido (2 epochs)
training_args = TrainingArguments(
    output_dir='./xlmr_email_ft',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=model_xlmr,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

print('Fine-tuning XLM-R (2 epochs)...')
trainer.train()

# Avaliação no dataset completo
model_xlmr.eval()
y_probs_xlmr = []
latencies_xlmr = []

print(f'\nInferência XLM-R ({len(texts)} amostras)...')
for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i:i+BATCH_SIZE]
    inputs = tokenizer_xlmr(
        batch_texts, return_tensors='pt', truncation=True,
        max_length=512, padding=True
    ).to(device)
    
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model_xlmr(**inputs)
    t1 = time.perf_counter()
    
    probs = torch.softmax(outputs.logits, dim=-1)
    phishing_probs = probs[:, 1].cpu().numpy()
    y_probs_xlmr.extend(phishing_probs)
    latencies_xlmr.append((t1 - t0) / len(batch_texts) * 1000)

y_probs_xlmr = np.array(y_probs_xlmr)
y_pred_xlmr = (y_probs_xlmr > 0.5).astype(int)

print(f'\n--- Resultados XLM-RoBERTa (fine-tuned) ---')
print(classification_report(y_true, y_pred_xlmr, target_names=['Legítimo', 'Phishing']))
print(f'AUC-ROC: {roc_auc_score(y_true, y_probs_xlmr):.4f}')
print(f'Latência média: {np.mean(latencies_xlmr):.2f} ms/amostra')
print(f'Latência P50: {np.percentile(latencies_xlmr, 50):.2f} ms/amostra')
print(f'Latência P95: {np.percentile(latencies_xlmr, 95):.2f} ms/amostra')

# Liberar memória
del model_xlmr, tokenizer_xlmr, trainer
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# 5. Testar Tradução PT->EN com MarianMT
# ============================================================
TRANSLATION_MODEL_ID = 'Helsinki-NLP/opus-mt-tc-big-pt-en'

tokenizer_marian = MarianTokenizer.from_pretrained(TRANSLATION_MODEL_ID)
model_marian = MarianMTModel.from_pretrained(TRANSLATION_MODEL_ID).to(device).eval()
print(f'Modelo de tradução carregado: {TRANSLATION_MODEL_ID}')

# Amostra manual de emails em português para teste de tradução
emails_pt = [
    {
        'subject': 'URGENTE: Sua conta será bloqueada',
        'body': ('Prezado cliente, detectamos atividade suspeita em sua conta. '
                 'Clique no link abaixo para verificar sua identidade imediatamente '
                 'ou sua conta será permanentemente bloqueada em 24 horas. '
                 'Link: http://banco-seguro-verificacao.com/login'),
        'expected': 'phishing'
    },
    {
        'subject': 'Confirmação de pedido #12345',
        'body': ('Olá, seu pedido foi confirmado com sucesso. '
                 'Prazo de entrega estimado: 5 dias úteis. '
                 'Para acompanhar seu pedido, acesse sua conta em nosso site. '
                 'Obrigado por comprar conosco!'),
        'expected': 'legitimate'
    },
    {
        'subject': 'Você ganhou um prêmio de R$ 50.000!',
        'body': ('Parabéns! Você foi selecionado para receber um prêmio especial. '
                 'Para resgatar seu prêmio, envie seus dados pessoais e bancários '
                 'para o email premios@sorteio-especial.com. '
                 'Oferta válida por 48 horas.'),
        'expected': 'phishing'
    },
    {
        'subject': 'Reunião de equipe - Sexta às 14h',
        'body': ('Oi pessoal, confirmando nossa reunião semanal de equipe '
                 'na sexta-feira às 14h na sala de conferências. '
                 'Pauta: revisão de sprint e planejamento da próxima iteração. '
                 'Tragam suas atualizações de status.'),
        'expected': 'legitimate'
    },
    {
        'subject': 'Atualização de segurança obrigatória',
        'body': ('Caro usuário, é necessário atualizar suas credenciais de segurança '
                 'imediatamente. Acesse http://seguranca-banco.tk/atualizar e informe '
                 'seu CPF, senha e número do cartão. Caso não atualize em 12 horas, '
                 'seu acesso será suspenso.'),
        'expected': 'phishing'
    }
]

# Carregar modelo DistilBERT para classificar traduções
EMAIL_MODEL_ID = 'cybersectony/phishing-email-detection-distilbert_v2.4.1'
tokenizer_email = AutoTokenizer.from_pretrained(EMAIL_MODEL_ID)
model_email = AutoModelForSequenceClassification.from_pretrained(EMAIL_MODEL_ID).to(device).eval()

print(f'\n{"="*60}')
print('Teste de Tradução PT->EN e Classificação')
print(f'{"="*60}\n')

translation_results = []
for email in emails_pt:
    full_text = f"Subject: {email['subject']}\n\n{email['body']}"
    
    # Traduzir
    t0 = time.perf_counter()
    inputs_tr = tokenizer_marian(
        full_text, return_tensors='pt', truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        translated_ids = model_marian.generate(**inputs_tr, max_length=512)
    translated_text = tokenizer_marian.decode(translated_ids[0], skip_special_tokens=True)
    t_translate = (time.perf_counter() - t0) * 1000
    
    # Classificar
    t0 = time.perf_counter()
    inputs_cls = tokenizer_email(
        translated_text, return_tensors='pt', truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        outputs_cls = model_email(**inputs_cls)
    probs = torch.softmax(outputs_cls.logits, dim=-1)
    phishing_prob = probs[0, 1].item()
    t_classify = (time.perf_counter() - t0) * 1000
    
    predicted = 'phishing' if phishing_prob > 0.5 else 'legitimate'
    correct = predicted == email['expected']
    
    translation_results.append({
        'subject': email['subject'],
        'expected': email['expected'],
        'predicted': predicted,
        'phishing_prob': phishing_prob,
        'correct': correct,
        'translate_ms': t_translate,
        'classify_ms': t_classify,
    })
    
    print(f'Subject: {email["subject"]}')
    print(f'  Tradução: {translated_text[:100]}...')
    print(f'  Esperado: {email["expected"]} | Predito: {predicted} ({phishing_prob:.2%})')
    print(f'  Correto: {"✓" if correct else "✗"} | Tradução: {t_translate:.0f}ms | Classificação: {t_classify:.0f}ms')
    print()

df_translation = pd.DataFrame(translation_results)
accuracy_pt = df_translation['correct'].mean()
print(f'\nAcurácia em emails PT (amostra manual): {accuracy_pt:.0%} ({df_translation["correct"].sum()}/{len(df_translation)})')
print(f'Latência média tradução: {df_translation["translate_ms"].mean():.0f}ms')
print(f'Latência média classificação: {df_translation["classify_ms"].mean():.0f}ms')

# Liberar modelo de tradução
del model_marian, tokenizer_marian
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# 6. Métricas — Tabela Comparativa
# ============================================================
results = []

for name, y_prob, y_pred, lats in [
    ('DistilBERT Email', y_probs_distilbert, y_pred_distilbert, latencies_distilbert),
    ('XLM-RoBERTa (ft)', y_probs_xlmr, y_pred_xlmr, latencies_xlmr),
]:
    results.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'AUC-ROC': roc_auc_score(y_true, y_prob),
        'Latência P50 (ms)': np.percentile(lats, 50),
        'Latência P95 (ms)': np.percentile(lats, 95),
    })

df_results = pd.DataFrame(results).set_index('Modelo')

def highlight_best(s):
    if 'Latência' in s.name:
        is_best = s == s.min()
    else:
        is_best = s == s.max()
    return ['background-color: #c6efce; font-weight: bold' if v else '' for v in is_best]

styled = (
    df_results.style
    .apply(highlight_best, axis=0)
    .format({
        'Accuracy': '{:.4f}',
        'Precision': '{:.4f}',
        'Recall': '{:.4f}',
        'F1-Score': '{:.4f}',
        'AUC-ROC': '{:.4f}',
        'Latência P50 (ms)': '{:.2f}',
        'Latência P95 (ms)': '{:.2f}',
    })
)
display(styled)

print('\n--- Confusion Matrix (DistilBERT) ---')
cm_distilbert = confusion_matrix(y_true, y_pred_distilbert)
print(cm_distilbert)

print('\n--- Confusion Matrix (XLM-RoBERTa) ---')
cm_xlmr = confusion_matrix(y_true, y_pred_xlmr)
print(cm_xlmr)

In [ ]:
# ============================================================
# 7. Comparação de Latência
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot de latências
ax = axes[0]
data_lat = [latencies_distilbert, latencies_xlmr]
labels_lat = ['DistilBERT\nEmail', 'XLM-RoBERTa\n(fine-tuned)']
bp = ax.boxplot(data_lat, labels=labels_lat, patch_artist=True)
colors = ['#2196F3', '#FF9800']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Latência (ms/amostra)')
ax.set_title('Distribuição de Latência por Modelo')
ax.grid(axis='y', alpha=0.3)

# Barras de latência P50/P95
ax = axes[1]
x = np.arange(2)
width = 0.35
p50 = [np.percentile(latencies_distilbert, 50), np.percentile(latencies_xlmr, 50)]
p95 = [np.percentile(latencies_distilbert, 95), np.percentile(latencies_xlmr, 95)]
ax.bar(x - width/2, p50, width, label='P50', color='#2196F3', alpha=0.8)
ax.bar(x + width/2, p95, width, label='P95', color='#f44336', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(['DistilBERT Email', 'XLM-RoBERTa (ft)'])
ax.set_ylabel('Latência (ms/amostra)')
ax.set_title('Latência P50 vs P95')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
os.makedirs('resultados', exist_ok=True)
plt.savefig('resultados/email_comparacao_latencia.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: resultados/email_comparacao_latencia.png')

In [ ]:
# ============================================================
# 8. Gráficos e Tabelas para o TCC
# ============================================================
os.makedirs('resultados', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- 8a. Matrizes de Confusão ---
for idx, (name, cm) in enumerate([
    ('DistilBERT Email', cm_distilbert),
    ('XLM-RoBERTa (ft)', cm_xlmr)
]):
    ax = axes[0][idx]
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Legítimo', 'Phishing'],
        yticklabels=['Legítimo', 'Phishing']
    )
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')
    ax.set_title(f'Matriz de Confusão — {name}')

# --- 8b. Curvas ROC ---
ax = axes[1][0]
for name, y_prob, color in [
    ('DistilBERT Email', y_probs_distilbert, '#2196F3'),
    ('XLM-RoBERTa (ft)', y_probs_xlmr, '#FF9800'),
]:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

# --- 8c. Comparação de Métricas (Barras) ---
ax = axes[1][1]
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
x = np.arange(len(metrics))
width = 0.35
vals_distilbert = [df_results.loc['DistilBERT Email', m] for m in metrics]
vals_xlmr = [df_results.loc['XLM-RoBERTa (ft)', m] for m in metrics]
ax.bar(x - width/2, vals_distilbert, width, label='DistilBERT Email', color='#2196F3', alpha=0.8)
ax.bar(x + width/2, vals_xlmr, width, label='XLM-RoBERTa (ft)', color='#FF9800', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=15)
ax.set_ylim(0.85, 1.01)
ax.set_ylabel('Score')
ax.set_title('Comparação de Métricas')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('resultados/email_benchmark_completo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: resultados/email_benchmark_completo.png')

# --- Tabela CSV ---
df_results.to_csv('resultados/email_benchmark_resultados.csv')
print('Salvo: resultados/email_benchmark_resultados.csv')

# --- Tabela de tradução ---
df_translation.to_csv('resultados/email_traducao_pt_en.csv', index=False)
print('Salvo: resultados/email_traducao_pt_en.csv')

print(f'\n{"="*60}')
print('Benchmark completo!')
print(f'{"="*60}')
print(f'\nResumo:')
print(f'  DistilBERT Email — F1: {df_results.loc["DistilBERT Email", "F1-Score"]:.4f}, AUC-ROC: {df_results.loc["DistilBERT Email", "AUC-ROC"]:.4f}')
print(f'  XLM-RoBERTa (ft) — F1: {df_results.loc["XLM-RoBERTa (ft)", "F1-Score"]:.4f}, AUC-ROC: {df_results.loc["XLM-RoBERTa (ft)", "AUC-ROC"]:.4f}')
print(f'  Tradução PT->EN — Acurácia: {accuracy_pt:.0%}')